# Sepsis population — analysis

<details>
<summary>Phase-2-only analysis of the virtual-patient sepsis cohort written by</summary>

`Sepsis_GenPopulation.ipynb` to `notebookData/population_sepsis_batch.h5`. No simulation runs
here: this notebook loads the converged cohort, derives physiological quantities, and
produces the paper figures/tables — the calibration-error violin/box grid, per-class
min/max LaTeX range tables, class-colored scatter panels, and dual-view (class + error)
scatter figures.

Every knob lives in the single `runConfig` cell (project rule: repo-root `CLAUDE.md`);
every code cell is region-wrapped (`.claude/rules/notebook-cells.md`).

</details>

In [ ]:
# region -> runConfig: the single run-configuration surface
# runConfig — the ONE place run config lives (project rule); Phase-2 analysis, no solve.
runConfig = {
    # --- input artifact (the Sepsis_GenPopulation.ipynb output) ------------
    "input": {"path": "notebookData/sepsis", "name": "population_sepsis_flatPCs.h5"},

    # --- analysis ----------------------------------------------------------
    "analysis": {
        "atm":               760.0,    # atmospheric offset for gauge conversion
        "divergenceLimit":   1e6,      # |obs| above this -> lane diverged, dropped
        "classToUse":        "class",  # class grouping: "class" | "class_extended"
        "errorVarSet":       "all",    # error grid group: "volume" | "pressure" | "all"
        "scatterOutlierPct": 100.0,    # drop scatter lane if any |rel error| exceeds this (%)
        # scenario whose volumeDistribution splits TBV across the compartments; None = the
        # one the artifact records in its `conf` attribute (the scenario the run used).
        "volumeScenario":    None,
    },

    # --- extended-class binning thresholds ---------------------------------
    "classBins": {"tbvSplit": 700.0, "coHigh": 3000.0, "coLow": 1750.0},  # TBV split, CO high/low bins

    # --- plotting ----------------------------------------------------------
    "plot": {
        "errorCap":    10.0,          # dual-view error colormap saturation (+/- %)
        "showFliers":  True,          # violin/box: draw box outliers (the published grid shows them)
        "ncols":       2,             # violin/box grid columns
        "cmap":        "coolwarm",    # dual-view error colormap
        # class-scatter var sets to build, one published figure each; a bare string is
        # accepted for a single set. "Resistance" -> Fig. 11, "ResRatio" -> 12, "Other" -> 13.
        "scatterVars": ["Resistance", "ResRatio", "Other"],
        "figScale":    1.0,           # global figure-size multiplier

        # cohort key, quoted in the paper captions: normal green, WS red, CS blue. The
        # published panels lay the cohorts out cold, norm, warm and name them that way,
        # so the artifact's phenotype names are ordered and relabelled to match.
        "classOrder":  ["coldShock", "normal", "warmShock"],
        "classLabels": {"coldShock": "cold", "normal": "norm", "warmShock": "warm"},
        "classColors": {"norm": "tab:green", "warm": "tab:red", "cold": "tab:blue"},

        # error grid (Fig. 9): panel order of the published figure -- the eight volumes, then
        # the pressures and flows -- plus the typography it was drawn at. Observations missing
        # from this list are appended in their artifact order unless `errorExclude` names them.
        "errorOrder": ["avg_V_As", "avg_V_Ap", "avg_V_Hl", "avg_V_Hr",
                       "avg_V_Cs", "avg_V_Vt", "avg_V_Cp", "avg_V_Vp",
                       "keep_max_P_As", "amp_P_As", "keep_max_P_Ap", "amp_P_Ap",
                       "avg_P_Vs", "CO", "avg_P_Cp", "avg_P_Cs"],
        # observations with no panel of their own: SV is reported as CO (= SV x HR), the
        # clinical quantity the published grid draws, and the two errors coincide.
        "errorExclude": ["keep_SV_Hl"],
        # panels the published figure titles by a name other than the signal's own label
        "errorLabelAlias": {"keep_max_P_As": "Sys_P_As", "keep_max_P_Ap": "Sys_P_Ap",
                            "avg_P_Vs": "CVP"},
        "errorFontSize":  48,         # panel titles; ticks 18 pt smaller, suptitle 6 pt larger
        "errorPanelSize": [7.6, 3.88],# inches per panel -- the published panel proportions
        "errorTickSpan":  0.0,        # panels framed tighter than this (%) get one decimal
        "errorTitle":     "Distribution of the relative error for the population targets",
        "errorTitleWrap": 40,         # suptitle wrap width in characters; 0 = one line
        # a one-line suptitle at this font is wider than the figure, and bbox_inches="tight"
        # then stretches the export sideways -- which is what wrapping keeps in proportion.

        # scatter panels (Figs. 11-13)
        "scatterFontSize":     38,
        "scatterTickFontSize": 30,
        "scatterMarkerSize":   75,
        "scatterPanelSize":    [15.0, 5.0],  # inches per panel
        # panel index carrying the cohort legend, per var set; None = none. The published
        # resistance figure legends its second panel, the other two caption the colour code.
        "scatterLegendPanel":  {"Resistance": 1, "ResRatio": None, "Other": None},
        # (x, y) per panel; each set is one published figure
        "scatterSets": {
            "Resistance": [["TotalBloodVolume", "Total Resistance"],
                           ["Cardiac Output", "Total Resistance"],
                           ["TotalBloodVolume", "Total Compliance"],
                           ["Cardiac Output", "Total Compliance"],
                           ["TotalBloodVolume", "Total systolic Elastance"],
                           ["TotalBloodVolume", "Total diastolic Elastance"]],
            "ResRatio":   [["Pressure Drop Artery", "R Ratio"], ["Pressure Drop Vein", "R Ratio"],
                           ["avg_P_Cs", "R_As_Cs"], ["avg_P_Cs", "R_Cs_Vs"]],
            "Other":      [["amp_P_As", "C_As"], ["avg_P_Cs", "C_Cs"],
                           ["keep_SV_Hl", "E_Hl"], ["avg_V_As", "V0_As"]],
        },

        # dual-view stack (Fig. 10): (parameter x, outcome y); the lower panel of each pair
        # is coloured by the outcome's relative error.
        "dualPairs": [["V0_Ap", "avg_V_Ap"], ["E_Hl", "keep_SV_Hl"], ["e_Hl", "avg_V_Hl"]],
        "dualPanelSize":      [15.0, 5.0],   # inches per panel (two panels per pair)
        "dualFontSize":       38,            # axis labels
        "dualTickFontSize":   20,
        "dualLegendFontSize": 20,
        "dualTitleFontSize":  30,
        "dualMarkerSize":     50,
        "dualCmapRange":      [0.005, 0.995],# colormap truncation, softening the extremes
        # grow each dual-view panel's upper x/y limit to the next whole tick when the cloud
        # overshoots the last one, so no points sit against an unlabelled edge.
        "dualPadTopToTick":   True,
    },

    # --- output ------------------------------------------------------------
    "output": {"saveLatex": True, "path": "notebookData/sepsis", "latexName": "sepsis_popanalysis_ranges.tex"},  # LaTeX range tables

    # --- paper artifacts (the figures main.tex \includegraphics from Images/) --------------
    # `emit` ships False: flip it True for one run to write the figures, then revert it.
    "paper": {
        "emit":          False,
        "imageDir":      "EFC_Paper/revision/Images",
        "errorFigure":   "PopulationErrorMerged_Smaller.png",    # Fig. 9
        "dualFigure":    "Error_V0Ap.png",                       # Fig. 10
        "scatterFigure": {"Resistance": "RES_Resistance.png",    # Fig. 11
                          "ResRatio":   "RES_RatioRes.png",      # Fig. 12
                          "Other":      "RES_RatioRes1.png"},    # Fig. 13
        "dpi":           100,
    },
}
# endregion

## Imports

In [ ]:
# region -> imports
# ---- repo-root bootstrap: run from any cwd (make `library` importable + resolve
# ---- the relative notebookData/ + config/ paths). Walks up to the dir containing library/. ----
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "library")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

import os
import json
import math
import textwrap

import numpy as np
import pandas as pd
import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns

import library.viz.plots as libPlots   # plotPopulationErrorGrid / plotPopulationScatter
import library.utils as utils

np.set_printoptions(suppress=True)
# endregion


## Load the cohort artifact

In [ ]:
# region -> load the cohort artifact (converged signals, targets, phenotype labels)
ATM = runConfig["analysis"]["atm"]
inPath = os.path.join(runConfig["input"]["path"], runConfig["input"]["name"])

def obsOffset(name):
    """Atmospheric offset baked into absolute-pressure signals (gauge = raw - offset)."""
    return ATM if ("_P_" in name and not name.startswith("amp_")) else 0.0

with h5py.File(inPath, "r") as f:
    modelStructure = json.loads(f["model_structure"].asstr()[()])
    sigNames       = list(f["raw_signal_names"].asstr()[:])
    observations   = list(f["observation_names"].asstr()[:])
    paramNames     = list(f["param_names"].asstr()[:])
    sampledParams  = np.asarray(f["sampled_params"][:])
    meta           = json.loads(f.attrs["meta"]) if "meta" in f.attrs else {}
    conf           = json.loads(f.attrs["conf"]) if "conf" in f.attrs else {}
    src            = "raw_coarse" if "raw_coarse" in f else "raw"
    lastRow        = np.asarray(f[src][:, -1, :])          # (N, C) run-end converged values

controlledParams = meta.get("controlledParams", [])
phenotypeIdx     = np.asarray(meta.get("phenotypeIdx", np.zeros(lastRow.shape[0], int)))
phenoNames       = meta.get("phenotypes", ["all"])
N                = lastRow.shape[0]

# per-lane converged model state (absolute units) — the feature-engineering source.
df_signals = pd.DataFrame(lastRow, columns=sigNames)

# gauge observation matrix + per-lane targets (target_<obs> columns of sampled_params).
sigIdx = {n: i for i, n in enumerate(sigNames)}
pnIdx  = {p: i for i, p in enumerate(paramNames)}
obsMatrix    = np.column_stack([lastRow[:, sigIdx[o]] - obsOffset(o) for o in observations])
targetMatrix = np.column_stack([sampledParams[:, pnIdx["target_" + o]] for o in observations])

# drop diverged / non-finite lanes (same rule as the generator).
divLimit  = runConfig["analysis"]["divergenceLimit"]
finiteRow = np.all(np.isfinite(obsMatrix), axis=1) & np.all(np.abs(obsMatrix) < divLimit, axis=1)
print(f"loaded {N} lanes from {inPath} | converged {int(finiteRow.sum())}/{N} | phenotypes {phenoNames}")
# endregion


## Derive physiological quantities + class labels

In [ ]:
# region -> derive physiological quantities + class labels
# Work on the converged lanes only, in the same order/space as the source notebook:
# derive everything from the absolute-unit signals, then convert pressures to gauge LAST.
pc       = runConfig["plot"]
keep     = finiteRow
df       = df_signals[keep].reset_index(drop=True).copy()
classIdx = phenotypeIdx[keep]
obsK     = obsMatrix[keep]
tgtK     = targetMatrix[keep]

# per-lane calibration error columns (signed relative %, and absolute).
relErr = (obsK - tgtK) / tgtK * 100.0
absErr = obsK - tgtK
for j, o in enumerate(observations):
    df[o + "_error_rel"] = relErr[:, j]
    df[o + "_error"]     = absErr[:, j]

# --- aggregate physiology (from the converged, still-absolute signals) ---------------
# Total blood volume. The eight targeted compartments each hold volumeDistribution[comp] x TBV,
# so their sum is the subject's blood volume scaled by the share they carry; dividing by that
# share recovers TBV. The systemic venous compartment is left out on purpose: this model
# calibrates C_Vs, not V_Vs, so V_Vs holds the conserved remainder of the twin's total rather
# than scaling with the subject, and including it pins every lane to the twin volume.
volScenario = runConfig["analysis"]["volumeScenario"] or conf.get("scenario")
volDist     = utils.loadScenario(volScenario)["shared"]["twin"]["volumeDistribution"]
volumeNames = [o for o in observations if o.startswith("avg_V_")]
volumeShare = sum(volDist[o[len("avg_V_"):]] for o in volumeNames)
df["TotalBloodVolume"] = df[volumeNames].sum(axis=1) / volumeShare
df["heart_rate"] = 60.0 / df["Cyc_HC"]

resistorNames = [n for n in sigNames if n.startswith("R_")]
df["Total Resistance"] = df[resistorNames].sum(axis=1)

capacitorNames = [n for n in sigNames if n.startswith("C_") and "_Vs" not in n]
df["Total Compliance"] = df[capacitorNames].sum(axis=1)

v0Names = [n for n in sigNames if n.startswith("V0_")]
df["Total UnstressedVolume"] = df[v0Names].sum(axis=1)

df["Perfusion Pressure"] = df["avg_P_As"] - df["avg_P_Vs"]
df["Perfusion Pressure Target"] = df["keep_min_P_As"] - df["avg_P_Vs"]

df["Mean Systemic Filling Pressure"] = df["TotalBloodVolume"] / df["Total Compliance"]
df["Venous Return"] = (df["Mean Systemic Filling Pressure"] - (df["avg_P_Vs"] - ATM)) / \
    (df["R_Cs_Vs"] + df["R_Vs_Vt"] + df["R_Vt_Hr"])
df["Venous Return*"] = (df["avg_P_As"] - df["P_Vt"]) / \
    (df["R_As_Cs"] + df["R_Cs_Vs"] + df["R_Vs_Vt"] + df["R_Vt_Hr"])

df["Pressure Drop Artery"] = ((df["keep_min_P_As"] - df["avg_P_Cs"]) / df["Perfusion Pressure Target"]) * 100
df["Pressure Drop Vein"]   = ((df["avg_P_Cs"] - df["avg_P_Vs"]) / df["Perfusion Pressure Target"]) * 100
df["Pressure Drop Artery*"] = ((df["avg_P_As"] - df["avg_P_Cs"]) / df["Perfusion Pressure"]) * 100
df["Pressure Drop Vein*"]   = ((df["avg_P_Cs"] - df["avg_P_Vs"]) / df["Perfusion Pressure"]) * 100

df["R Ratio"] = df["R_As_Cs"] / (df["R_Cs_Vs"] + df["R_As_Cs"])

df["E Ratio"]  = df["E_Hl"] / df["E_Hr"]
df["El Ratio"] = df["E_Hl"] / df["e_Hl"]
df["Er Ratio"] = df["E_Hr"] / df["e_Hr"]
df["Total systolic Elastance"]  = df["E_Hl"] + df["E_Hr"]
df["Total diastolic Elastance"] = df["e_Hl"] + df["e_Hr"]
df["Total diastolic Compliance"] = 1.0 / df["Total diastolic Elastance"]
df["E_As"] = 1.0 / df["C_As"]; df["LvA Ratio"] = df["E_As"] / df["E_Hl"]
df["E_Ap"] = 1.0 / df["C_Ap"]; df["RvAp Ratio"] = df["E_Hr"] / df["E_Ap"]

df["Cardiac Output"] = df["keep_SV_Hl"] * df["heart_rate"]
df["Shock index"] = df["heart_rate"] / df["keep_max_P_As"]   # keep_max_P_As still absolute here

# Derived clinical target: the published error grid reports CO, not the stroke volume the
# controller targets. HR is prescribed per lane (the Cyc_HC driver), so CO = SV x HR on both
# the observed and the target side and the two relative errors coincide.
hr    = df["heart_rate"].to_numpy()
iSV   = observations.index("keep_SV_Hl")
coObs, coTgt = obsK[:, iSV] * hr, tgtK[:, iSV] * hr
df["CO_error_rel"] = (coObs - coTgt) / coTgt * 100.0
df["CO_error"]     = coObs - coTgt

# --- class labels (from meta phenotypeIdx; extended = TBV x CO bins) ------------------
# Cohorts are ordered and renamed to the published key (cold, norm, warm) so every figure
# lays them out and legends them the way the paper does.
classOrder  = [c for c in pc["classOrder"] if c in phenoNames]
classOrder += [c for c in phenoNames if c not in classOrder]
classLabels = pc["classLabels"]
dispClass   = lambda c: classLabels.get(c, c)
df["class"] = pd.Categorical([dispClass(phenoNames[i]) for i in classIdx],
                             categories=[dispClass(c) for c in classOrder])
tbvSplit = runConfig["classBins"]["tbvSplit"]
coHigh   = runConfig["classBins"]["coHigh"]
coLow    = runConfig["classBins"]["coLow"]
normalName = dispClass(phenoNames[0])
ext = []
for i in range(len(df)):
    base = df["class"].iloc[i]
    if base == normalName:
        ext.append(str(base))
    else:
        vol = df["TotalBloodVolume"].iloc[i]; co = df["Cardiac Output"].iloc[i]
        volTag = "norV" if vol > tbvSplit else "lowV"
        coTag = "highC" if co > coHigh else ("lowC" if co < coLow else "normC")
        ext.append(f"{base}_{coTag}_{volTag}")
df["class_extended"] = pd.Categorical(ext, categories=sorted(set(ext)))

# --- convert absolute pressures to gauge (done LAST, like the source) ----------------
bareP = [c for c in ["P_As", "P_Cs", "P_Vs", "P_Vt", "P_Hr", "P_Ap", "P_Cp", "P_Vp", "P_Hl"]
         if c in df.columns]
df[bareP] = df[bareP] - ATM
pressure_cols = [c for c in df.columns if "_P_" in c and "amp" not in c and "error" not in c]
df[pressure_cols] = df[pressure_cols] - ATM

classNames = list(df["class"].cat.categories)
print(f"{len(df)} converged subjects | class counts:\n{df['class'].value_counts().to_string()}")
print(f"TBV from {len(volumeNames)} targeted compartments carrying {volumeShare:.2f} of "
      f"{volScenario} volumeDistribution")
print(f"TBV {df['TotalBloodVolume'].min():.1f}-{df['TotalBloodVolume'].max():.1f} mL | "
      f"CO {df['Cardiac Output'].min():.0f}-{df['Cardiac Output'].max():.0f} mL/min")
# endregion

## Figure 1 — calibration-error distributions

In [ ]:
# region -> Figure 1: calibration-error violin+box small-multiples grid (Fig. 9)
classToUse  = runConfig["analysis"]["classToUse"]
errorVarSet = runConfig["analysis"]["errorVarSet"]

# published panel order first (some panels are derived, e.g. CO), then any observation it
# does not name and `errorExclude` does not deliberately drop.
order     = [o for o in pc["errorOrder"] if o + "_error_rel" in df.columns]
order    += [o for o in observations if o not in order and o not in pc["errorExclude"]]
volumes   = [o for o in order if "_V_" in o]
panels    = {"volume":   volumes,
             "pressure": [o for o in order if o not in volumes],
             "all":      order}[errorVarSet]

cats   = list(df[classToUse].cat.categories)
colors = pc["classColors"] if set(cats) <= set(pc["classColors"]) else None
title  = ("\n".join(textwrap.wrap(pc["errorTitle"], width=pc["errorTitleWrap"]))
          if pc["errorTitleWrap"] else pc["errorTitle"])

figError = libPlots.plotPopulationErrorGrid(
    np.column_stack([df[o + "_error_rel"].to_numpy() for o in panels]),
    df[classToUse].astype(str).to_numpy(), panels,
    classOrder=cats, classColors=colors, labelAlias=pc["errorLabelAlias"],
    ncols=pc["ncols"], panelSize=tuple(pc["errorPanelSize"]), fontSize=pc["errorFontSize"],
    showFliers=pc["showFliers"], tickSpan=pc["errorTickSpan"], title=title)
plt.show()
# endregion

## Table — per-class parameter / observation ranges

In [ ]:
# region -> per-class min/max range tables (LaTeX via utils.generate_latex_table_new)
saveLatex = runConfig["output"]["saveLatex"]
classCol  = "class"
classCats = list(df[classCol].cat.categories)

def min_max_rows(frame, columns, decimals):
    """{labelFor(col): ['[min, max]' per class]} for generate_latex_table_new."""
    fmt = f"{{:.{decimals}f}}"
    groups = {c: frame[frame[classCol] == c] for c in classCats}
    rows = {}
    for col in columns:
        rows[utils.labelFor(col, "latex")] = [
            f"[{fmt.format(g[col].min())}, {fmt.format(g[col].max())}]" for g in groups.values()]
    return rows

paramCols = [p for p in controlledParams if p in df.columns]
obsCols   = [o for o in observations if o in df.columns]

latex_params = utils.generate_latex_table_new(
    min_max_rows(df, paramCols, 2), ["Parameter"] + classCats, "", "", "sepsisParamRanges",
    f"Calibrated parameter ranges per class across {len(df)} virtual subjects.")
latex_obs = utils.generate_latex_table_new(
    min_max_rows(df, obsCols, 1), ["Observation"] + classCats, "", "", "sepsisObsRanges",
    f"Observation ranges per class across {len(df)} virtual subjects.")

print(latex_params); print(); print(latex_obs)
if saveLatex:
    outTex = os.path.join(runConfig["output"]["path"], runConfig["output"]["latexName"])
    with open(outTex, "w") as fh:
        fh.write(latex_params + "\n\n" + latex_obs)
    print(f"\nLaTeX ranges written to: {outTex}")
# endregion


## Figure 2 — class-colored scatter panels

In [ ]:
# region -> Figure 2: class-colored scatter panels (Figs. 11-13)
scatterVars = pc["scatterVars"]
scatterVars = [scatterVars] if isinstance(scatterVars, str) else list(scatterVars)
outlierPct  = runConfig["analysis"]["scatterOutlierPct"]
legendPanel = pc["scatterLegendPanel"]

errCols = [o + "_error_rel" for o in observations]
mask    = ~df[errCols].abs().gt(outlierPct).any(axis=1)
df_f    = df[mask]
print(f"{df_f.shape[0]}/{len(df)} lanes within +/-{outlierPct}% error")
print(df_f["class"].value_counts().to_string())

figScatter = {}
for varSet in scatterVars:
    pairs = [tuple(p) for p in pc["scatterSets"][varSet]]
    figScatter[varSet] = libPlots.plotPopulationScatter(
        {n: df_f[n].to_numpy() for pair in pairs for n in pair}, pairs,
        df_f["class"].astype(str).to_numpy(),
        classOrder=classNames, classColors=pc["classColors"],
        panelSize=tuple(pc["scatterPanelSize"]), fontSize=pc["scatterFontSize"],
        tickFontSize=pc["scatterTickFontSize"], markerSize=pc["scatterMarkerSize"],
        legendPanel=(legendPanel.get(varSet) if isinstance(legendPanel, dict) else legendPanel))
plt.show()
# endregion

## Figure 3 — dual-view (class + relative-error) scatter

In [ ]:
# region -> Figure 3: dual-view (class + relative-error) scatter (Fig. 10)
pairs = [tuple(p) for p in pc["dualPairs"]]

figDual = libPlots.plotPopulationDualView(
    {n: df[n].to_numpy() for pair in pairs for n in pair}, pairs,
    df["class"].astype(str).to_numpy(),
    {y: df[y + "_error_rel"].to_numpy() for _, y in pairs},
    classOrder=classNames, classColors=pc["classColors"],
    panelSize=tuple(pc["dualPanelSize"]), fontSize=pc["dualFontSize"],
    tickFontSize=pc["dualTickFontSize"], legendFontSize=pc["dualLegendFontSize"],
    titleFontSize=pc["dualTitleFontSize"], markerSize=pc["dualMarkerSize"],
    errorCap=pc["errorCap"], cmapName=pc["cmap"], cmapRange=tuple(pc["dualCmapRange"]),
    padTopToTick=pc["dualPadTopToTick"])
plt.show()
# endregion

## Emit the paper figures

In [ ]:
# region -> emit the paper figures into EFC_Paper/revision/Images (guarded by paper.emit)
paper = runConfig["paper"]
if paper["emit"]:
    os.makedirs(paper["imageDir"], exist_ok=True)
    figures = [(figError, paper["errorFigure"]), (figDual, paper["dualFigure"])]
    figures += [(fig, paper["scatterFigure"][varSet]) for varSet, fig in figScatter.items()]
    for fig, name in figures:
        if fig is None:
            continue
        out = os.path.join(paper["imageDir"], name)
        fig.savefig(out, dpi=paper["dpi"], bbox_inches="tight")
        print(f"wrote {out}")
else:
    print("paper.emit is False -- figures not written")
# endregion